In [9]:
import jax.numpy as np
import jax
from jax import vmap, jit, grad, jacfwd, jacrev

import jax
import jax.numpy as jnp
import tensorflow as tf
import tensorflow_datasets as tfds
import neural_tangents as nt
import orbax.checkpoint

import ml_collections
import matplotlib.pyplot as plt

from train import *
from compute_ntk import *

import jax.flatten_util

In [4]:
def get_config():
  """Get the default hyperparameter configuration."""
  config = ml_collections.ConfigDict()

  config.learning_rate = 0.001
  config.latents = 20
  config.batch_size = 128
  config.num_epochs = 200
  return config

def load_state(checkpointer, file):
   """ Load a model state. """
   state = checkpointer.restore(file)

   return state["model"]

def get_dataset():
  ds_builder = tfds.builder('binarized_mnist')
  ds_builder.download_and_prepare()

  return input_pipeline.build_train_set(config.batch_size, ds_builder)

def get_apply_fn(config):

    def apply_fn(params, inputs):
        rng = jax.random.key(0)
        rng, init_rng = jax.random.split(rng)
        return models.model(config.latents).apply(
            {"params": params}, inputs, rng
        )[0]

    return apply_fn

def compute_cvs(ntk):

    eigs, _ = np.linalg.eigh(ntk)

    eigs = np.clip(eigs, 1e-11, None)

    eigs /= eigs.sum()

    return -np.sum(eigs * np.log(eigs)), np.trace(ntk)

  
# Get the config
config = get_config()
orbax_checkpointer = orbax.checkpoint.StandardCheckpointer()
# Set rng
rng = random.key(0)
rng, key = random.split(rng)

# Load the dataset
train_dataset = get_dataset()
test_ds = next(train_dataset)[0:4]

model = models.model(config.latents)
apply_fn = get_apply_fn(config)
state = load_state(orbax_checkpointer, "/work/stovey/novely-model-study/vae/model_14")


2024-04-23 13:23:18.738725: W external/xla/xla/service/gpu/nvptx_compiler.cc:718] The NVIDIA driver's CUDA version is 12.3 which is older than the ptxas CUDA version (12.4.131). Because the driver is older than the ptxas version, XLA is disabling parallel compilation, which may slow down compilation. You should update your NVIDIA driver or use the NVIDIA-provided CUDA forward compatibility packages.


In [19]:
jacobian_fn = jit(jacfwd(apply_fn, argnums=0))


In [20]:
jc_1 = jacobian_fn(state['params'], test_ds[0])
# jc_2 = jacobian_fn(state['params'], test_ds[2:])


In [21]:
jax.flatten_util.ravel_pytree(jc_1)[0]

Array([-0.01129913, -0.09399414,  0.        , ...,  0.        ,
        0.        ,  0.        ], dtype=float32)